In [ ]:
# model_info.py
import os, sys, re, hashlib, platform, time, shutil, subprocess, textwrap
from pathlib import Path

MODEL_PATH = Path(r"D:\NCM Project\ALLaM-7B-Instruct-preview-Q8_0.gguf")  # <-- عدّل المسار هنا

# ---------- Helpers ----------
def human_bytes(n):
    for unit in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def file_hash(path, algo="sha256", chunk=1024*1024):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def guess_quant(file_name: str):
    # أمثلة شائعة: Q8_0, Q6_K, Q5_K_M, Q4_K_M, Q3_K_L, Q2_K, etc.
    m = re.search(r"(Q\d(?:_\d)?(?:_K(?:_[A-Z])?)?)", file_name.upper())
    return m.group(1) if m else "Unknown"

def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, shell=True, text=True)
        return out.strip()
    except Exception as e:
        return f"ERROR: {e}"

# ---------- Basic file info ----------
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
print("="*80)
print("MODEL FILE")
print("- Path     :", MODEL_PATH)
print("- Exists   :", True)
print("- Size     :", human_bytes(MODEL_PATH.stat().st_size))
print("- MD5      :", file_hash(MODEL_PATH, "md5"))
print("- SHA256   :", file_hash(MODEL_PATH, "sha256"))
print("- Quant    :", guess_quant(MODEL_PATH.name))
print("="*80)

# ---------- System / GPU info ----------
print("SYSTEM")
print("- OS       :", platform.platform())
print("- Python   :", sys.version.split()[0])
print("- CPU      :", platform.processor() or platform.machine())

# nvidia-smi / nvcc (لو عندك CUDA)
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    print("- nvidia-smi:\n", run_cmd("nvidia-smi"))
else:
    print("- nvidia-smi: NOT FOUND")

nvcc = shutil.which("nvcc")
print("- nvcc     :", run_cmd("nvcc --version") if nvcc else "NOT FOUND")
print("="*80)

# ---------- GGUF metadata (اختياري) ----------
 try:
    from gguf import GGUFReader  # pip install gguf
    print("GGUF METADATA (from file)")
    rdr = GGUFReader(str(MODEL_PATH))
    # مفاتيح وقيَم الميتاداتا
    kv = rdr.meta_data  # dict[str, Any]
    # اطبع أهم المفاتيح إن وُجدت
    keys_of_interest = [
        "general.architecture",
        "general.file_type",
        "general.quantization_version",
        "general.name",
        "vocab.type",
        "vocab.size",
        "tokenizer.ggml.model",
        "tokenizer.chat_template",
        "llama.context_length",
        "llama.embedding_length",
        "llama.block_count",
        "llama.attention.head_count",
        "llama.attention.head_count_kv",
        "llama.attention.layer_norm_rms_epsilon",
        "llama.rope.dimension_count",
        "llama.rope.freq_base",
        "gguf.version",
    ]
    for k in keys_of_interest:
        if k in kv:
            print(f"- {k:40s}: {kv[k]}")
    # عدد التنسورات
    try:
        print(f"- tensors_count: {len(rdr.tensors)}")
    except Exception:
        pass
except Exception as e:
    print("GGUF METADATA: gguf package not available or failed to read.")
    print("Tip: pip install gguf")
print("="*80)

# ---------- Try via llama-cpp-python (vocab/info + benchmark) ----------
# ملاحظة: هذا الجزء يحمل الموديل (قد يستغرق ثوانٍ/دقائق حسب جهازك)
def bench_with_llama_cpp():
    try:
        from llama_cpp import Llama  # pip install llama-cpp-python
    except Exception as e:
        print("LLAMA-CPP: package not installed. Tip: pip install llama-cpp-python")
        return

    # إعدادات خفيفة للفحص الأولي: حمل فقط المفردات vocab لو تبغى سرعة
    try:
        print("LLAMA-CPP: Loading model (vocab_only=True) to get quick info...")
        llm = Llama(model_path=str(MODEL_PATH), vocab_only=True)
        # الـ bindings الحديثة تعطيك معلومات سياقية:
        try:
            ctx = llm._ctx  # تحذير: خاص داخلي وقد يتغير بين الإصدارات
            # بعض الخصائص قد لا تتوفر دائمًا، لذلك نحوط بـ try/except
            print("- vocab_only loaded OK.")
        except Exception:
            pass
    except Exception as e:
        print("LLAMA-CPP (vocab_only) FAILED:", e)
        # نكمل بمحاولة التحميل العادي لو فشل
    print("-"*80)

    # تحميل عادي مع توليد قصير + قياس سرعة التوكن/ثانية
    try:
        print("LLAMA-CPP: Loading model (normal) for a short benchmark...")
        t0 = time.time()
        llm = Llama(
            model_path=str(MODEL_PATH),
            n_ctx=1024,
            n_threads=os.cpu_count() or 4,
            n_gpu_layers=0,   # غيّرها >0 لو عندك CUDA build
        )
        t1 = time.time()
        print(f"- Load time: {t1 - t0:.2f} sec")

        prompt = "اكتب سطر اختبار قصير."
        gen_tokens = 64
        t0 = time.time()
        out = llm(prompt, max_tokens=gen_tokens, temperature=0.6, top_p=0.9)
        t1 = time.time()

        text = out["choices"][0]["text"]
        elapsed = t1 - t0
        # تقريب عدد التوكنات الناتجة (قد لا يوفّر العدد بدقة)
        produced = len(text.split())
        tps = produced / elapsed if elapsed > 0 else 0
        print("- Output sample:", repr(text.strip()[:160]))
        print(f"- Gen time: {elapsed:.2f} sec | approx tokens: ~{produced} | ~{tps:.2f} tok/s (rough)")
    except Exception as e:
        print("LLAMA-CPP (benchmark) FAILED:", e)

bench_with_llama_cpp()

print("="*80)
print("DONE.")


MODEL FILE
- Path     : D:\NCM Project\ALLaM-7B-Instruct-preview-Q8_0.gguf
- Exists   : True
- Size     : 6.93 GB
- MD5      : fbd7d5c60975c854722b0df3db9bfa8d
- SHA256   : 952b26b3fd4edb6511ef79a655384dc31ef7a093c9ded4899ef383deea09f2b4
- Quant    : Q8_0
SYSTEM
- OS       : Windows-10-10.0.26100-SP0
- Python   : 3.11.9
- CPU      : Intel64 Family 6 Model 158 Stepping 10, GenuineIntel
- nvidia-smi:
 Wed Aug 13 14:25:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 561.19                 Driver Version: 561.19         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|============

llama_model_loader: loaded meta data with 31 key-value pairs and 291 tensors from D:\NCM Project\ALLaM-7B-Instruct-preview-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = ALLaM AI ALLaM 7B Instruct Preview
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-preview
llama_model_loader: - kv   4:                           general.basename str              = ALLaM-AI-ALLaM
llama_model_loader: - kv   5:                         general.size_label str              = 7B
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama

LLAMA-CPP: Loading model (vocab_only=True) to get quick info...
- vocab_only loaded OK.
--------------------------------------------------------------------------------
LLAMA-CPP: Loading model (normal) for a short benchmark...


llama_model_loader: - kv  12:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv  13:                 llama.attention.head_count u32              = 32
llama_model_loader: - kv  14:              llama.attention.head_count_kv u32              = 32
llama_model_loader: - kv  15:                       llama.rope.freq_base f32              = 10000.000000
llama_model_loader: - kv  16:     llama.attention.layer_norm_rms_epsilon f32              = 0.000010
llama_model_loader: - kv  17:                           llama.vocab_size u32              = 64000
llama_model_loader: - kv  18:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv  19:                       tokenizer.ggml.model str              = llama
llama_model_loader: - kv  20:                         tokenizer.ggml.pre str              = default
llama_model_loader: - kv  21:                      tokenizer.ggml.tokens arr[str,64000]   = ["<unk>", "<s>", "</s>

- Load time: 0.98 sec


llama_perf_context_print:        load time =    1322.80 ms
llama_perf_context_print: prompt eval time =    1322.44 ms /     5 tokens (  264.49 ms per token,     3.78 tokens per second)
llama_perf_context_print:        eval time =   33451.73 ms /    63 runs   (  530.98 ms per token,     1.88 tokens per second)
llama_perf_context_print:       total time =   34837.30 ms /    68 tokens


- Output sample: ''
- Gen time: 34.84 sec | approx tokens: ~0 | ~0.00 tok/s (rough)
DONE.
